In [128]:
# Import python modules
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)

project_root = os.path.abspath(os.path.join(".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [129]:
# Now this works because Python sees `src` as a top-level module
from src import analytics, plotting, utils

In [130]:
import yaml

In [131]:
# TODO: Read both no_bat and bat results. Make a copy of the config with co2-prices

In [132]:
BASE_FOLDER = os.path.dirname(os.getcwd())
RUNS_FOLDER = os.path.join(BASE_FOLDER, "runs")
BATCH_RUNS_FOLDER = os.path.join(RUNS_FOLDER, "batch_runs")

DATA_FOLDER = os.path.join(BASE_FOLDER, "data")
FOR_THE_REPORT_FOLDER = os.path.join(DATA_FOLDER, "for_the_report")
METHODOLOGY_FOLDER = os.path.join(FOR_THE_REPORT_FOLDER, "methodology")
BENCHMARKING_FOLDER = os.path.join(METHODOLOGY_FOLDER, "benchmarking")
os.makedirs(BENCHMARKING_FOLDER, exist_ok=True)

In [133]:
co2_prices = [80, 100, 120, 140, 160]
emissions_restriction_reduction = [0.0, 0.20, 0.40, 0.60, 0.80, 1.0]

In [134]:
folder_names = [
    "May16_Fri_h20_m37_s52-GTSEP_stochastic_v1-37_ES_PT_no_bat",
    "May16_Fri_h20_m41_s08-GTSEP_stochastic_v1-37_ES_PT",
]
config_names = ["config_37_sv1_2_weeks_no_bats", "config_37_sv1_2_weeks"]

In [135]:
SINGLE_RUNS_FOLDER = os.path.join(RUNS_FOLDER, "single_runs")

# Switch to batch runs folder when doing the 128 4week model run

In [136]:
folders = [
    os.path.join(SINGLE_RUNS_FOLDER, folder_name) for folder_name in folder_names
]
print(folders)

['c:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\runs\\single_runs\\May16_Fri_h20_m37_s52-GTSEP_stochastic_v1-37_ES_PT_no_bat', 'c:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\runs\\single_runs\\May16_Fri_h20_m41_s08-GTSEP_stochastic_v1-37_ES_PT']


## Read data

In [137]:
dummy_co2_limit = 100

In [138]:
configs = [utils.load_model_config(config_name) for config_name in config_names]
for config in configs:
    print(config)

Path_or_name: config_37_sv1_2_weeks_no_bats
Configuration loaded from c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats.yaml
Path_or_name: config_37_sv1_2_weeks
Configuration loaded from c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks.yaml
{'data_folder_name': 'elec_s_37_ES_PT_no_bat', 'model_name': 'GTSEP_stochastic_v1', 'model_id': '37_ES_PT_no_bat', 'MIPGap': 0.01, 'VOLL': 6350, 'CC': 100, 'CO2_price': 80, 'E_limit': inf, 'p_max_new_branch': 5000, 'p_min_new_branch': 100, 'expansion_factor': 2.0, 'MS': 0.1, 'years': [2025, 2030, 2040, 2050], 'clustering_unit': None, 'clustering_periods': None, 'scenario_file': None, 'EVPI': False, 'VSS': False, 'battery_capacity_path': None, 'generator_capacity_path': None, 'branch_capacity_path': None, 'co2_emissions_path': None, 'query': 'not (x > 2 and y < 40)', 'carriers': ['CCGT', 'solar', 'onwind'], 'discount_rate': 0.07, 'representa

In [139]:
years_list = [config["years"] for config in configs]
data_folder_names = [config["data_folder_name"] for config in configs]
for i in range(len(configs)):
    print(f"Config {i}:")
    print(f"Data folder path: {data_folder_names[i]}")
    print(f"Years: {years_list[i]}")

Config 0:
Data folder path: elec_s_37_ES_PT_no_bat
Years: [2025, 2030, 2040, 2050]
Config 1:
Data folder path: elec_s_37_ES_PT_no_bat_limit
Years: [2025, 2030, 2040, 2050]


In [140]:
input_data_folders = [
    os.path.join(DATA_FOLDER, "processed", folder_name)
    for folder_name in data_folder_names
]
input_data_sets = [
    utils.load_multi_year_csv_files_with_week_from_folder(
        years=years, data_folder_path=input_data_folder
    )
    for years, input_data_folder in zip(years_list, input_data_folders)
]
# for i in range(len(input_data_sets)):
#     print(f"config: {i}, {config_names[i]}")
#     print(f"Input data folder: {input_data_folders[i]}")
#     print(f"Input data set: {input_data_sets[i]}")

In [141]:
folders = [
    os.path.join(SINGLE_RUNS_FOLDER, folder_name) for folder_name in folder_names
]
print(folders)

['c:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\runs\\single_runs\\May16_Fri_h20_m37_s52-GTSEP_stochastic_v1-37_ES_PT_no_bat', 'c:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\runs\\single_runs\\May16_Fri_h20_m41_s08-GTSEP_stochastic_v1-37_ES_PT']


In [142]:
decision_variables_folders = [
    os.path.join(folder, "decision_variables") for folder in folders
]
decision_variables_sets = [
    utils.load_csv_files_from_folder_with_scenarios(decision_variables_folder)
    for decision_variables_folder in decision_variables_folders
]
for i in range(len(decision_variables_sets)):
    print(f"config: {i}, {config_names[i]}")
    print(f"Decision variables set: {decision_variables_sets[i]}")

config: 0, config_37_sv1_2_weeks_no_bats
Decision variables set: {'battery_capacity': Empty DataFrame
Columns: [value]
Index: [], 'battery_charging': Empty DataFrame
Columns: [value]
Index: [], 'battery_discharging': Empty DataFrame
Columns: [value]
Index: [], 'battery_soc': Empty DataFrame
Columns: [value]
Index: [], 'branch_capacity':              value
branch year       
35     2025    0.0
       2030    0.0
       2040    0.0
       2050    0.0, 'curtailment':                                      value
generator   scenario year week hour       
ES1 0 CCGT  NT       2025 21   0       0.0
                               1       0.0
                               2       0.0
                               3       0.0
                               4       0.0
...                                    ...
PT1 0 solar DE       2050 42   163     0.0
                               164     0.0
                               165     0.0
                               166     0.0
               

In [143]:
decision_variables_sets[0].keys()

dict_keys(['battery_capacity', 'battery_charging', 'battery_discharging', 'battery_soc', 'branch_capacity', 'curtailment', 'generation', 'generator_capacity', 'load_shedding', 'power_flow'])

In [144]:
# Finally, calculate the CO2 emissions for each scenario, but first I need the scenarios
scenario_file_path = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\scenario_multipliers\base_scenarios.csv"
import pandas as pd

In [145]:
scenarios_set = []
Omegas = []
week_weights_set = []
configs_after_running = []
for i, config in enumerate(configs):
    scenario_file = config["scenario_file"]
    print(f"Scenario file: {scenario_file}")
    scenario_multiplier = utils.load_scenario_multiplier(scenario_file)

    # Check that years in scenario_multiplier match the years in the data
    scenario_years = scenario_multiplier.index.values.tolist()
    scenarios_list = scenario_multiplier.columns.tolist()
    scenarios = {
        year: [name for name in scenario_multiplier.loc[year].dropna().index]
        for year in scenario_multiplier.index
    }
    scenarios_set.append(scenarios)

    Omegas.append(scenarios_list)
    config_after_running = yaml.safe_load(
        open(os.path.join(folders[i], "model_info", "config.yaml"))
    )
    print(f"Config after running: {config_after_running}")
    configs_after_running.append(config_after_running)
    week_weights = config_after_running["week_weights"]
    week_weights_set.append(week_weights)

    Omega = scenarios_list  # e.g. ['NT','GA','DE']
    Omega_y = scenarios  # e.g. {2040:['NT','GA','DE'],...}

    # Still need Ys, Gs, Ws, and Ts
for i, scenario in enumerate(scenarios_set):
    print(f"Scenario: {scenario}")
    print(f"Omega: {Omegas[i]}")
    print(week_weights_set[i])

Scenario file: None
Config after running: {'CC': 100, 'CO2_price': 80, 'EVPI': False, 'E_limit': inf, 'MIPGap': 0.01, 'MS': 0.1, 'VOLL': 6350, 'VSS': False, 'battery_capacity_path': None, 'branch_capacity_path': None, 'carriers': ['CCGT', 'solar', 'onwind'], 'clustering_periods': None, 'clustering_unit': None, 'data_folder_name': 'elec_s_37_ES_PT_no_bat', 'discount_rate': 0.07, 'expansion_factor': 2.0, 'generator_capacity_path': None, 'model_id': '37_ES_PT_no_bat', 'model_name': 'GTSEP_stochastic_v1', 'p_max_new_branch': 5000, 'p_min_new_branch': 100, 'query': 'not (x > 2 and y < 40)', 'representative_period_unit': 'week', 'representative_periods': [21, 42], 'run_id': 'May16_Fri_h20_m37_s52-GTSEP_stochastic_v1-37_ES_PT_no_bat', 'save_folder': 'C:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\runs\\single_runs\\May16_Fri_h20_m37_s52-GTSEP_stochastic_v1-37_ES_PT_no_bat', 'scenario_file': None, 'week_weights': {21: 26.071428571428573, 42: 26.071428571428573}, 'years

In [146]:
decision_variables_sets[0].keys()

dict_keys(['battery_capacity', 'battery_charging', 'battery_discharging', 'battery_soc', 'branch_capacity', 'curtailment', 'generation', 'generator_capacity', 'load_shedding', 'power_flow'])

In [147]:
import pandas as pd
import os


def compute_total_co2_emissions(
    decision_variables: dict[str, pd.DataFrame],
    generators: pd.DataFrame,
    week_weights: dict[str, float],
    savefolder: str | None = None,
) -> pd.DataFrame:
    """
    Compute total CO₂ emissions per year and scenario, preserving NaNs
    where no data exists.

    Emissions are calculated as:
      sum_over(i,w,t) [ value(i,ω,y,w,t) * co2_emissions(i,y) * week_weights[w] ]

    Parameters
    ----------
    decision_variables : dict of pd.DataFrame
        Contains 'generation' DataFrame with MultiIndex
        ['generator','scenario','year','week','hour'] and column 'value'.
    generators : pd.DataFrame
        MultiIndexed by ['year','generator'], with column 'co2_emissions'.
    week_weights : dict of str->float or int->float
        Mapping week -> weight.
    savefolder : str or None
        Directory to save CSV. If None, not saved.

    Returns
    -------
    pd.DataFrame
        Indexed by year, columns=scenario, values=emissions (tonnes),
        with NaN for missing year+scenario combinations.
    """
    # Convert week_weights keys to int if needed
    ww_int = {int(k): v for k, v in week_weights.items()}
    # Flatten generation
    gen = (
        decision_variables["generation"]
        .reset_index()
        .rename(columns={"value": "gen_mwh"})
    )
    # Merge CO₂ factors
    meta = generators.reset_index()[["year", "generator", "co2_emissions"]]
    df = gen.merge(meta, on=["year", "generator"], how="left")
    # Map week → weight
    df["weight"] = df["week"].map(ww_int)
    # Compute emissions
    df["emissions_tonnes"] = df["gen_mwh"] * df["co2_emissions"] * df["weight"]
    # Aggregate and pivot without filling NaNs
    table = (
        df.groupby(["year", "scenario"])["emissions_tonnes"]
        .sum()
        .reset_index()
        .pivot(index="year", columns="scenario", values="emissions_tonnes")
    )
    if savefolder:
        table.to_csv(os.path.join(savefolder, "annual_co2_emissions.csv"))
    return table

In [148]:
co2_emissions_dataframes = []
for i, decision_variables_set in enumerate(decision_variables_sets):
    print(f"Folder name: {folder_names[i]}")
    # print(f"Decision variables: {decision_variables_set}")
    co2_emissions = compute_total_co2_emissions(
        decision_variables=decision_variables_set,
        generators=input_data_sets[i]["generators"],
        week_weights=week_weights_set[i],
        savefolder=BENCHMARKING_FOLDER,
    )
    co2_emissions_dataframes.append(co2_emissions)
    print(co2_emissions)

Folder name: May16_Fri_h20_m37_s52-GTSEP_stochastic_v1-37_ES_PT_no_bat
scenario            DE            GA            NT
year                                              
2025               NaN           NaN  1.260497e+07
2030               NaN           NaN  1.103644e+07
2040      1.872068e+07  1.372133e+07  1.758866e+07
2050      2.274106e+07  1.668874e+07           NaN
Folder name: May16_Fri_h20_m41_s08-GTSEP_stochastic_v1-37_ES_PT
scenario            DE            GA            NT
year                                              
2025               NaN           NaN  1.300753e+07
2030               NaN           NaN  1.166625e+07
2040      1.846069e+07  1.314106e+07  1.725016e+07
2050      2.161876e+07  1.502774e+07           NaN


In [149]:
input_data_sets[0]["generators"].head()

bus     carrier         p_nom  marginal_cost  \
year generator                                                          
2025 ES1 0 CCGT        ES1 0        CCGT  26305.332000       40.77238   
     ES1 0 coal        ES1 0        coal   4739.393483       28.19697   
     ES1 0 offwind-ac  ES1 0  offwind-ac   9274.463486        0.01500   
     ES1 0 onwind      ES1 0      onwind  26825.862669        0.01500   
     ES1 0 ror         ES1 0         ror    147.008247        0.00000   

                        capital_cost  co2_emissions    color  \
year generator                                                 
2025 ES1 0 CCGT         99027.729293           0.20  #b20101   
     ES1 0 coal        349976.553630           0.34  #707070   
     ES1 0 offwind-ac  184160.119456           0.00  #6895dd   
     ES1 0 onwind       96085.888020           0.00  #235ebc   
     ES1 0 ror         299140.224929           0.00  #4adbc8   

                                nice_name  extendable  extension_potential  \
year generator                                                               
2025 ES1 0 CCGT        Combined-Cycle Gas        True                  0.0   
     ES1 0 coal                      Coal        True                  0.0   
     ES1 0 offwind-ac  Offshore Wind (AC)        True                  0.0   
     ES1 0 onwind            Onshore Wind        True                  0.0   
     ES1 0 ror               Run of River        True                  0.0   

                       extended_by  
year generator                      
2025 ES1 0 CCGT                0.0  
     ES1 0 coal                0.0  
     ES1 0 offwind-ac          0.0  
     ES1 0 onwind              0.0  
     ES1 0 ror                 0.0

In [150]:
co2_emissions_folder = os.path.join(DATA_FOLDER, "co2_emissions_files")
os.makedirs(co2_emissions_folder, exist_ok=True)
for i, co2_emissions in enumerate(co2_emissions_dataframes):
    co2_emissions_path = os.path.join(
        co2_emissions_folder, f"co2_emissions_{folder_names[i]}.csv"
    )
    co2_emissions.to_csv(co2_emissions_path)

In [151]:
co2_emissions_folder = os.path.join(DATA_FOLDER, "co2_emissions_files")
test_read_dfs = []
co2_emissions_paths = []
os.makedirs(co2_emissions_folder, exist_ok=True)
for i, co2_emissions in enumerate(co2_emissions_dataframes):
    co2_emissions_path = os.path.join(
        co2_emissions_folder, f"co2_emissions_{folder_names[i]}.csv"
    )
    test_read_dfs.append(pd.read_csv(co2_emissions_path, index_col=0))
    co2_emissions_paths.append(co2_emissions_path)

In [152]:
test1 = test_read_dfs[0]
test1

,DE,GA,NT
year,,,
2025,NaN,NaN,1.260497e+07
2030,NaN,NaN,1.103644e+07
2040,1.872068e+07,1.372133e+07,1.758866e+07
2050,2.274106e+07,1.668874e+07,NaN


In [153]:
years = [2025, 2030, 2040, 2050]
scenarios = ["DE", "GA", "NT"]
for year in years:
    for scenario in scenarios:

        print(
            f"For year {year} scenario {scenario} the value is {test1.loc[year, scenario]}"
        )
        print(type(test1.loc[year, scenario]))

For year 2025 scenario DE the value is nan
<class 'numpy.float64'>
For year 2025 scenario GA the value is nan
<class 'numpy.float64'>
For year 2025 scenario NT the value is 12604966.388761766
<class 'numpy.float64'>
For year 2030 scenario DE the value is nan
<class 'numpy.float64'>
For year 2030 scenario GA the value is nan
<class 'numpy.float64'>
For year 2030 scenario NT the value is 11036443.84543883
<class 'numpy.float64'>
For year 2040 scenario DE the value is 18720680.92616463
<class 'numpy.float64'>
For year 2040 scenario GA the value is 13721325.62371502
<class 'numpy.float64'>
For year 2040 scenario NT the value is 17588655.904235482
<class 'numpy.float64'>
For year 2050 scenario DE the value is 22741056.579966318
<class 'numpy.float64'>
For year 2050 scenario GA the value is 16688743.386459729
<class 'numpy.float64'>
For year 2050 scenario NT the value is nan
<class 'numpy.float64'>


# Time to make copies of the config files

In [154]:
for i, name in enumerate(folder_names):
    for num, co2_price in enumerate(co2_prices):
        suffix = f"_co2price_{co2_price}"
        overwrite_dict = {
            "co2_price": co2_price,
        }
        utils.copy_and_modify_config(
            config_names[i], overwrite_dict=overwrite_dict, suffix=suffix
        )

New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_80.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_100.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_120.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_140.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_160.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_co2price_80.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_

In [155]:
for i, name in enumerate(folder_names):
    for num, co2_price in enumerate(co2_prices):
        suffix = f"_co2price_{co2_price}"
        overwrite_dict = {
            "co2_price": co2_price,
        }
        utils.copy_and_modify_config(
            config_names[i], overwrite_dict=overwrite_dict, suffix=suffix
        )

New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_80.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_100.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_120.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_140.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_co2price_160.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_co2price_80.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_

In [156]:
for i, name in enumerate(folder_names):
    for num, emission_restriction in enumerate(emissions_restriction_reduction):
        suffix = f"_emissionsreduction_{emission_restriction}"
        co2_emissions_dataframe = co2_emissions_dataframes[i]
        co2_emissions_dataframe_scaled = co2_emissions_dataframe * (
            1 - emission_restriction
        )
        co2_emissions_path = os.path.join(
            co2_emissions_folder, f"co2_emissions_{folder_names[i]}{suffix}.csv"
        )
        co2_emissions_dataframe_scaled.to_csv(co2_emissions_path)
        overwrite_dict = {
            "co2_emissions_path": co2_emissions_path,
        }
        utils.copy_and_modify_config(
            config_names[i], overwrite_dict=overwrite_dict, suffix=suffix
        )

New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_emissionsreduction_0.0.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_emissionsreduction_0.2.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_emissionsreduction_0.4.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_emissionsreduction_0.6.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_emissionsreduction_0.8.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_sv1_2_weeks_no_bats_emissionsreduction_1.0.yaml
New config saved to: c:\Users\tinus\OneDrive\Dokumen